In [ ]:
from collections import defaultdict
from gensim import corpora, models
from csv import QUOTE_NONE
from pathlib import Path
import os
import datetime as dt
import pandas as pd
import nltk

#nltk.download()

In [ ]:
# Load school shooters' texts

shooter_paths = dict()
for csv in (Path(os.path.abspath("")).parents[1] / "data" / "school_shooters").rglob("data.csv"):
    shooter_paths[csv.parents[0].name] = csv

shooter_dfs = dict()
for key, value in shooter_paths.items():
    shooter_dfs[key] = pd.read_csv(value, encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)

shooter_dfs

In [ ]:
from nltk.corpus import stopwords
import re

def preprocess_document(text):

    def process_word(word):
        # Some basic preprocessing...
        # Could maybe implement preprocessing of hashtags as well later

        sw = stopwords.words("english")
        url_re = re.compile(r'(http|www)(\S+)?', re.IGNORECASE)
        punctuation_re = r'[^\w\s]'
        is_url = lambda x: True if url_re.search(x) is not None else False

        word = "urlhyperlink" if is_url(word) else word
        word = re.sub(punctuation_re, "", word)

        if word in sw or (len(word) < 3):
            word = ""

        return word

    text = nltk.word_tokenize(text.lower(), preserve_line=False)

    preprocessed_text = []
    for word in text:
        processed_word = process_word(word)
        if processed_word != "" and processed_word != "urlhyperlink":
            preprocessed_text.append(processed_word)

    return preprocessed_text

In [ ]:
# Fetching all texts and filtering / preprocessing
import pprint

preprocessed_texts = []
for df in shooter_dfs.values():
    for text in df["text"]:
        preprocessed_texts.append(preprocess_document(text))

print(preprocessed_texts)

""" cho_df = pd.DataFrame(shooter_dfs["Seung_Hui_Cho"])
cho_texts = cho_df["text"].values """

In [ ]:
# Remove one-off words

frequency = defaultdict(int)
for text in preprocessed_texts:
    for token in text:
        frequency[token] += 1

preprocessed_texts = [[token for token in text if frequency[token] > 1] for text in preprocessed_texts]

dictionary = corpora.Dictionary(preprocessed_texts)
corpus = [dictionary.doc2bow(text) for text in preprocessed_texts]

In [ ]:
from gensim import models

n_topics = 10                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

lda_model = models.LdaModel(corpus, num_topics=n_topics, id2word = dictionary)
lda_model.show_topics(num_words=5)

In [ ]:
""" 
tfidf = models.TfidfModel(corpus)
lsi_model = models.LsiModel(corpus_tfidf, id2word=dictionary, num_topics=n_topics)
corpus_lsi = lsi_model[corpus_tfidf]

lsi_model.print_topics(n_topics) """